In [ ]:
import sympy as sp
from sympy import *  # noqa
from sympy.solvers.solveset import linsolve

sp.init_session()
sp.init_printing()

### Global variables:

In [ ]:
# Dimensional parameters
comps = 6
dd = comps
DD = comps

# Speed of light, time, time step
c = sp.symbols(r"c", real=True, positive=True)
t = sp.symbols(r"t", real=True)
tn = sp.symbols(r"t_n", real=True)
dt = sp.symbols(r"\Delta{t}", real=True, positive=True)

# Components of k vector, omega, norm of k vector
kx = sp.symbols(r"k_x", real=True)
ky = sp.symbols(r"k_y", real=True)
kz = sp.symbols(r"k_z", real=True)
om = sp.symbols(r"omega", real=True, positive=True)
knorm = sp.sqrt(kx**2 + ky**2 + kz**2)

### Method:
The goal is to solve a system of second-order ordinary differential equations in time of the form $\boldsymbol{\ddot{X}} = M \boldsymbol{X}$, where the $d \times d$ matrix $M$ has eigenpairs $(\lambda_i, \{\boldsymbol{v}_{i,1}, \dots, \boldsymbol{v}_{i,\mu_i}\})$, where $\mu_i$ denotes the algebraic multiplicity of $\lambda_i$ and $\sum_i \mu_i = d$.
Assuming that all eigenvalues are either zero or negative, we can write $\lambda_i = - \omega_i^2$, with $\omega_i = \sqrt{- \lambda_i} \geq 0$.
Then the solution of $\boldsymbol{\ddot{X}} = M \boldsymbol{X}$ reads
$$
\boldsymbol{X} = \sum_i \sum_j^{\mu_i} \left(a_{i,j} C(\omega_i, t) + b_{i,j} S(\omega_i, t) \right) \boldsymbol{v}_{i,j} \,,
$$
where $a_{i,j}$ and $b_{i,j}$ are integration constants to be determined by the initial conditions $\boldsymbol{X}(t_n)$ and $\boldsymbol{\dot{X}}(t_n)$, $C(\omega, t) = \cos(\omega \, (t - t_n))$, and
$$
S(\omega, t) =
\begin{cases}
(t - t_n) & \omega = 0 \,, \\
\dfrac{\sin(\omega \, (t - t_n))}{\omega} & \omega \neq 0 \,.
\end{cases}
$$
We remark that
$$
\begin{aligned}
& \boldsymbol{X}(t_n) = \sum_i \sum_j^{\mu_i} \left(a_{i,j} C(\omega_i, t_n) + b_{i,j} S(\omega_i, t_n) \right) \boldsymbol{v}_{i,j} = \sum_i \sum_j^{\mu_i} a_{i,j} \boldsymbol{v}_{i,j} \,, \\
& \boldsymbol{\dot{X}}(t_n) = \sum_i \sum_j^{\mu_i} \left(a_{i,j} \dot{C}(\omega_i, t_n) + b_{i,j} \dot{S}(\omega_i, t_n) \right) \boldsymbol{v}_{i,j} = \sum_i \sum_j^{\mu_i} b_{i,j} \boldsymbol{v}_{i,j} \,.
\end{aligned}
$$
In fact, the second time derivative of $\boldsymbol{X}$ yields
$$
\begin{split}
\boldsymbol{\ddot{X}} & = \sum_i \sum_j^{\mu_i} \left(a_{i,j} \ddot{C}(\omega_i, t) + b_{i,j} \ddot{S}(\omega_i, t) \right) \boldsymbol{v}_{i,j} \\
& = \sum_i (-\omega_i^2) \sum_j^{\mu_i} \left(a_{i,j} C(\omega_i, t) + b_{i,j} S(\omega_i, t) \right) \boldsymbol{v}_{i,j} = M \boldsymbol{X} \,,
\end{split}
$$
where we used the fact that, by definition, $M \boldsymbol{v}_{i,j} = \lambda_i \boldsymbol{v}_{i,j} = - \omega_i^2 \boldsymbol{v}_{i,j}$ for all $j = 1, \dots, \mu_i$.

### Auxiliary functions:

In [ ]:
def C(omega, t):
    return sp.cos(omega * (t - tn))


def S(omega, t):
    return (t - tn) if omega == 0.0 else sp.sin(omega * (t - tn)) / omega


def Xt(eigenpairs, a, b, t):
    """
    Compute X(t) according to the formulas above
    for a given set of eigenpairs and coefficients.
    """
    XX = zeros(DD, 1)
    # Index used for the integration constants a_n and b_n
    i = 0
    # Loop over matrix eigenpairs
    for ep in eigenpairs:
        # ep[0] is an eigenvalue and om = sp.sqrt(-ep[0])
        omega = 0.0 if ep[0] == 0.0 else om
        # am is the algebraic multiplicity of the eigenvalue
        am = ep[1]
        # vF is the list of all eigenvectors corresponding to the eigenvalue
        vX = ep[2]
        # Loop over algebraic multiplicity of the eigenvalue
        for j in range(am):
            XX += (a[i] * C(omega, t) + b[i] * S(omega, t)) * vX[j]
            i += 1
    return XX


def evolve(X, dX_dt, d2X_dt2):
    """
    Solve ordinary differential equation X'' = M*X.
    """
    # Set matrix for given ODE
    M = zeros(DD)
    for i in range(DD):
        for j in range(DD):
            M[i, j] = d2X_dt2[i].coeff(X[j], 1)

    print("M:")
    display(M)

    print("Verifying (d2X_dt2 - MX)...")
    diff = d2X_dt2 - M * X
    diff = diff.subs(t, tn).subs(om, c * knorm).expand()
    diff = simplify(diff)
    display(diff)

    # Characteristic matrix
    lamda = sp.symbols(r"lamda")
    Id = eye(DD)
    M_charmat = M - lamda * Id

    # Characteristic polynomial
    M_charpoly = M_charmat.det()
    M_charpoly = factor(M_charpoly.as_expr())

    print(r"Characteristic polynomial:")
    display(M_charpoly)

    M_eigenvals = sp.solve(M_charpoly, lamda)

    # List of eigenvectors
    M_eigenvects = []

    # List of eigenpairs
    M_eigenpairs = []

    # Compute eigenvectors as null spaces
    for ev in M_eigenvals:
        # M - lamda * Id
        A = M_charmat.subs(lamda, ev)
        A.simplify()

        print(r"Eigenvalue:")
        display(ev)

        print(r"Characteristic matrix:")
        display(A)

        # Compute null space and store eigenvectors
        v = A.nullspace()
        M_eigenvects.append(v)

        print(r"Eigenvectors:")
        display(v)

        # Store eigenpairs (eigenvalue, algebraic multiplicity, eigenvectors)
        M_eigenpairs.append((ev, len(v), v))

    # Verify that the eigenpairs satisfy the characteristic equations
    for ep in M_eigenpairs:
        for j in range(ep[1]):
            print(f"Verifying characteristic equation for {ep[0]} and {ep[2][j]}...")
            diff = M * ep[2][j] - ep[0] * ep[2][j]
            diff = sp.simplify(diff)
            display(diff)

    # Define integration constants
    a = []
    b = []
    for i in range(DD):
        an = r"a_{:d}".format(i + 1)
        bn = r"b_{:d}".format(i + 1)
        a.append(sp.symbols(an))
        b.append(sp.symbols(bn))

    # Set equations corresponding to initial conditions
    lhs_a = Xt(M_eigenpairs, a, b, tn) - X
    lhs_b = Xt(M_eigenpairs, a, b, t).diff(t).subs(t, tn) - dX_dt

    # Compute integration constants from initial conditions
    # (convert list of tuples to list using list comprehension)
    a = list(linsolve(list(lhs_a), a))
    a = [item for el in a for item in el]
    b = list(linsolve(list(lhs_b), b))
    b = [item for el in b for item in el]

    # Evaluate solution at t = tn + dt
    X_new = Xt(M_eigenpairs, a, b, tn + dt).expand()
    for d in range(DD):
        for Eij in E:
            X_new[d] = X_new[d].collect(Eij)
        for Bij in B:
            X_new[d] = X_new[d].collect(Bij)

    # Check correctness by taking *second* derivative
    # and comparing with initial right-hand side at time tn
    print("Verifying (d2X_dt2 - X_t.diff)...")
    X_t = Xt(M_eigenpairs, a, b, t)
    diff = d2X_dt2 - X_t.diff(t).diff(t).subs(t, tn).subs(om, c * knorm).expand()
    diff = sp.simplify(diff)
    display(diff)

    return X_t, X_new

In [ ]:
# indices  0     1     2     3     4     5
labels = ["xy", "xz", "yx", "yz", "zx", "zy"]

# E fields
Exy = sp.symbols(r"E_{xy}")
Exz = sp.symbols(r"E_{xz}")
Eyx = sp.symbols(r"E_{yx}")
Eyz = sp.symbols(r"E_{yz}")
Ezx = sp.symbols(r"E_{zx}")
Ezy = sp.symbols(r"E_{zy}")
E = Matrix([[Exy], [Exz], [Eyx], [Eyz], [Ezx], [Ezy]])

# B fields
Bxy = sp.symbols(r"B_{xy}")
Bxz = sp.symbols(r"B_{xz}")
Byx = sp.symbols(r"B_{yx}")
Byz = sp.symbols(r"B_{yz}")
Bzx = sp.symbols(r"B_{zx}")
Bzy = sp.symbols(r"B_{zy}")
B = Matrix([[Bxy], [Bxz], [Byx], [Byz], [Bzx], [Bzy]])

# dE/dt
dExy_dt = c**2 * I * ky * (Bzx + Bzy)
dExz_dt = -(c**2) * I * kz * (Byx + Byz)
dEyx_dt = -(c**2) * I * kx * (Bzx + Bzy)
dEyz_dt = c**2 * I * kz * (Bxy + Bxz)
dEzx_dt = c**2 * I * kx * (Byx + Byz)
dEzy_dt = -(c**2) * I * ky * (Bxy + Bxz)
dE_dt = Matrix(
    [
        [dExy_dt],
        [dExz_dt],
        [dEyx_dt],
        [dEyz_dt],
        [dEzx_dt],
        [dEzy_dt],
    ]
)

# dB/dt
dBxy_dt = -I * ky * (Ezx + Ezy)
dBxz_dt = I * kz * (Eyx + Eyz)
dByx_dt = I * kx * (Ezx + Ezy)
dByz_dt = -I * kz * (Exy + Exz)
dBzx_dt = -I * kx * (Eyx + Eyz)
dBzy_dt = I * ky * (Exy + Exz)
dB_dt = Matrix(
    [
        [dBxy_dt],
        [dBxz_dt],
        [dByx_dt],
        [dByz_dt],
        [dBzx_dt],
        [dBzy_dt],
    ]
)

# d2E/dt2
d2Exy_dt2 = c**2 * I * ky * (dBzx_dt + dBzy_dt)
d2Exz_dt2 = -(c**2) * I * kz * (dByx_dt + dByz_dt)
d2Eyx_dt2 = -(c**2) * I * kx * (dBzx_dt + dBzy_dt)
d2Eyz_dt2 = c**2 * I * kz * (dBxy_dt + dBxz_dt)
d2Ezx_dt2 = c**2 * I * kx * (dByx_dt + dByz_dt)
d2Ezy_dt2 = -(c**2) * I * ky * (dBxy_dt + dBxz_dt)
d2E_dt2 = Matrix(
    [
        [d2Exy_dt2],
        [d2Exz_dt2],
        [d2Eyx_dt2],
        [d2Eyz_dt2],
        [d2Ezx_dt2],
        [d2Ezy_dt2],
    ]
)

# d2B/dt2
d2Bxy_dt2 = -I * ky * (dEzx_dt + dEzy_dt)
d2Bxz_dt2 = I * kz * (dEyx_dt + dEyz_dt)
d2Byx_dt2 = I * kx * (dEzx_dt + dEzy_dt)
d2Byz_dt2 = -I * kz * (dExy_dt + dExz_dt)
d2Bzx_dt2 = -I * kx * (dEyx_dt + dEyz_dt)
d2Bzy_dt2 = I * ky * (dExy_dt + dExz_dt)
d2B_dt2 = Matrix(
    [
        [d2Bxy_dt2],
        [d2Bxz_dt2],
        [d2Byx_dt2],
        [d2Byz_dt2],
        [d2Bzx_dt2],
        [d2Bzy_dt2],
    ]
)

for i in range(dd):
    d2E_dt2[i] = sp.expand(d2E_dt2[i])

for i in range(dd):
    d2B_dt2[i] = sp.expand(d2B_dt2[i])

print("dE_dt:")
display(dE_dt)
print("dB_dt:")
display(dB_dt)
print("d2E_dt2:")
display(d2E_dt2)
print("d2B_dt2:")
display(d2B_dt2)

### Solve second-order ODEs for $\boldsymbol{E}$, $\boldsymbol{B}$, $F$ and $G$:

In [ ]:
print(r"Solve Equations for E:")
E_t, E_new = evolve(E, dE_dt, d2E_dt2)

# print(r"Solve Equations for B:")
# B_t, B_new = evolve(B, dB_dt, d2B_dt2)

# # Check correctness by taking *first* derivative
# # and comparing with initial right-hand side at time tn
# # E
# diff = E_t.diff(t).subs(t, tn).subs(om, c * knorm).expand() - dE_dt
# diff.simplify()
# if diff != zeros(DD, 1):
#     print(rf"Could Not Verify Time Integration for {str(E)}")
#     display(diff)
# # B
# diff = B_t.diff(t).subs(t, tn).subs(om, c * knorm).expand() - dB_dt
# diff.simplify()
# if diff != zeros(DD, 1):
#     print(rf"Could Not Verify Time Integration for {str(B)}")
#     display(diff)

### Coefficients of PSATD equations in PML:

In [ ]:
# #      0,   1,   2,   3,   4,   5
# # E: Exy, Exz, Eyx, Eyz, Ezx, Ezy
# # B: Bxy, Bxz, Byx, Byz, Bzx, Bzy

# # Select update equation (left hand side)
# X_new = sp.Matrix.vstack(E_new, B_new)
# for i in range(X_new.shape[0]):
#     field_lhs = X_new[i, 0]
#     # Extract individual terms (right hand side)
#     X = sp.Matrix.vstack(E, B)
#     for j in range(X.shape[0]):
#         field_rhs = X[j, 0]
#         coeff = field_lhs.coeff(field_rhs, 1).simplify()
#         print(rf"Coefficient of {str(field_rhs)} Multiplying {str(field_rhs)}")
#         display(coeff)
#         # print(ccode(Assignment(sp.symbols(r'LHS'), C1)))